# ML Assignment 2 — guided project notebook

Run cells in order from your **ML_Assignment2** folder. This notebook contains the full Python source and can recreate `app.py`. Training regenerates models and report files; back up any README edits first.

The notebook is a development companion. GitHub and Streamlit still need `app.py`, dependencies, test CSV and saved model artifacts. Your final assignment submission is a separate PDF with real links, BITS Lab proof and full README. Read and adapt the code and observations in accordance with the assignment's AI learning-support rule.


## 1. Environment
Your required packages are already installed. Check the current working directory before continuing.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image
PROJECT = Path.cwd()
print("Project folder:", PROJECT)
print("Python:", sys.version)


## 2. Full project source
This cell stores the complete training and Streamlit application source. Run it to create `app.py`. You can inspect and edit the source inside this cell. Existing `app.py` will be overwritten.

In [ ]:
APP_SOURCE = '"""Learning companion for ML Assignment 2 (read and adapt before submission).\n\nINSTALL: python -m pip install numpy pandas scikit-learn matplotlib joblib streamlit\nTRAIN:   python app.py --train\nAPP:     python -m streamlit run app.py\nJUPYTER: %run app.py --train\n\nUses sklearn\'s packaged copy of UCI Wisconsin Diagnostic Breast Cancer data.\nOptionally train from the original UCI file: --train --data data/wdbc.data\nAll generated files live beside this script. Training overwrites generated outputs.\nRun training on BITS Lab too and capture your own execution screenshot.\n"""\nfrom pathlib import Path\nimport argparse\nimport json\nimport platform\nimport sys\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.datasets import load_breast_cancer\nfrom sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.neighbors import KNeighborsClassifier\nfrom sklearn.naive_bayes import GaussianNB\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,\n                             recall_score, f1_score, matthews_corrcoef,\n                             confusion_matrix, classification_report)\n\nROOT = Path(__file__).resolve().parent\nSOURCE = \'https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic\'\nSEED = 42\nTARGET = \'target\'\nSLUGS = {\'Logistic Regression\': \'logistic_regression\', \'Decision Tree\': \'decision_tree\',\n         \'KNN\': \'knn\', \'Naive Bayes\': \'naive_bayes\', \'Support Vector Machine\': \'svm\'}\n\n\ndef dataset(path=None):\n    """Use consistent column names and malignant=1 for either data source."""\n    reference = load_breast_cancer(as_frame=True)\n    columns = list(reference.data.columns)\n    if path:\n        raw = pd.read_csv(path, header=None)\n        if raw.shape != (569, 32):\n            raise ValueError(\'Expected original wdbc.data: 569 rows, 32 columns, no header.\')\n        X = raw.iloc[:, 2:].copy()\n        X.columns = columns\n        y = raw.iloc[:, 1].map({\'M\': 1, \'B\': 0})\n    else:\n        X = reference.data.copy()\n        # sklearn\'s native mapping is malignant=0; explicitly reverse it.\n        y = 1 - reference.target\n    y = y.rename(TARGET)\n    if y.isna().any() or X.isna().any().any():\n        raise ValueError(\'Unexpected missing values or diagnosis labels.\')\n    assert X.shape[0] >= 500 and X.shape[1] >= 12\n    return X, y\n\n\ndef models():\n    # Each pipeline fits scaling only on its training partition, including in CV.\n    return {\n        \'Logistic Regression\': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED)),\n        \'Decision Tree\': make_pipeline(DecisionTreeClassifier(max_depth=5, min_samples_leaf=3, random_state=SEED)),\n        \'KNN\': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),\n        \'Naive Bayes\': make_pipeline(StandardScaler(), GaussianNB()),\n        \'Support Vector Machine\': make_pipeline(StandardScaler(), SVC(C=1.0, kernel=\'rbf\', probability=True, random_state=SEED)),\n    }\n\n\ndef evaluate(model, X, y):\n    predictions = model.predict(X)\n    positive_index = list(model.classes_).index(1)\n    probabilities = model.predict_proba(X)[:, positive_index]\n    values = {\n        \'Accuracy\': accuracy_score(y, predictions),\n        \'AUC\': roc_auc_score(y, probabilities) if len(np.unique(y)) == 2 else np.nan,\n        \'Precision\': precision_score(y, predictions, zero_division=0),\n        \'Recall\': recall_score(y, predictions, zero_division=0),\n        \'F1\': f1_score(y, predictions, zero_division=0),\n        \'MCC\': matthews_corrcoef(y, predictions),\n    }\n    return values, predictions, probabilities\n\n\ndef markdown_table(frame):\n    # No optional tabulate dependency required.\n    return \'| \' + \' | \'.join(frame.columns) + \' |\\n| \' + \' | \'.join([\'---\'] * len(frame.columns)) + \' |\\n\' + \'\\n\'.join(\n        \'| \' + \' | \'.join(f\'{v:.4f}\' if isinstance(v, (float, np.floating)) else str(v) for v in row) + \' |\'\n        for row in frame.itertuples(index=False, name=None))\n\n\ndef train(data_path=None):\n    X, y = dataset(data_path)\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)\n    for folder in [\'model\', \'results\', \'data\']:\n        (ROOT / folder).mkdir(exist_ok=True)\n    pd.concat([X, y], axis=1).to_csv(ROOT / \'data\' / \'dataset.csv\', index=False)\n    pd.concat([X_test, y_test], axis=1).to_csv(ROOT / \'test_data.csv\', index=False)\n    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)\n    rows, validation = [], []\n    for name, model in models().items():\n        # Compare predefined models on training-only CV; do not tune on test data.\n        scores = cross_validate(model, X_train, y_train, cv=cv, scoring={\'f1\': \'f1\', \'auc\': \'roc_auc\'})\n        validation.append({\'ML Model Name\': name, \'CV F1\': scores[\'test_f1\'].mean(), \'CV AUC\': scores[\'test_auc\'].mean()})\n        model.fit(X_train, y_train)\n        values, predicted, _ = evaluate(model, X_test, y_test)\n        rows.append({\'ML Model Name\': name, **values})\n        joblib.dump(model, ROOT / \'model\' / f\'{SLUGS[name]}.joblib\')\n        report = classification_report(y_test, predicted, labels=[0, 1], target_names=[\'Benign\', \'Malignant\'], zero_division=0, output_dict=True)\n        (ROOT / \'results\' / f\'{SLUGS[name]}_report.json\').write_text(json.dumps(report, indent=2))\n        import matplotlib\n        matplotlib.use(\'Agg\')\n        import matplotlib.pyplot as plt\n        from sklearn.metrics import ConfusionMatrixDisplay\n        fig, ax = plt.subplots(figsize=(5, 4))\n        ConfusionMatrixDisplay(confusion_matrix(y_test, predicted, labels=[0, 1]), display_labels=[\'Benign\', \'Malignant\']).plot(ax=ax, cmap=\'Blues\', colorbar=False)\n        ax.set_title(name)\n        fig.tight_layout()\n        fig.savefig(ROOT / \'results\' / f\'{SLUGS[name]}_confusion.png\', dpi=150)\n        plt.close(fig)\n    metrics = pd.DataFrame(rows)\n    cv_results = pd.DataFrame(validation).sort_values([\'CV F1\', \'CV AUC\'], ascending=False)\n    winner = cv_results.iloc[0][\'ML Model Name\']\n    metrics.to_csv(ROOT / \'results\' / \'metrics.csv\', index=False)\n    cv_results.to_csv(ROOT / \'results\' / \'cross_validation.csv\', index=False)\n    metadata = {\'features\': list(X.columns), \'target\': TARGET, \'mapping\': {\'0\': \'Benign\', \'1\': \'Malignant\'},\n                \'train_rows\': len(X_train), \'test_rows\': len(X_test), \'seed\': SEED,\n                \'sklearn_version\': sklearn.__version__, \'python_version\': platform.python_version(),\n                \'selected_model\': winner, \'source\': SOURCE}\n    (ROOT / \'model\' / \'metadata.json\').write_text(json.dumps(metadata, indent=2))\n    # Pin installed versions so cloud loads artifacts with matching sklearn.\n    from importlib.metadata import version\n    dependencies = [\'numpy\', \'pandas\', \'scikit-learn\', \'matplotlib\', \'joblib\', \'streamlit\']\n    lines = []\n    for package in dependencies:\n        try:\n            lines.append(f\'{package}=={version(package)}\')\n        except Exception:\n            lines.append(package)\n    (ROOT / \'requirements.txt\').write_text(\'\\n\'.join(lines) + \'\\n\')\n    explanations = {\n        \'Logistic Regression\': \'Linear baseline; scaling supports optimization. Examine whether a linear boundary is sufficient.\',\n        \'Decision Tree\': \'Captures nonlinear rules; depth and leaf limits reduce overfitting. Compare training-only validation with held-out results.\',\n        \'KNN\': \'Distance-based classifier; scaling prevents large-scale features dominating distances.\',\n        \'Naive Bayes\': \'Gaussian feature likelihoods; correlated nucleus measurements may limit its conditional-independence assumption.\',\n        \'Support Vector Machine\': \'Scaled RBF kernel can model nonlinear boundaries; probabilities provide the AUC input.\'}\n    observations = []\n    for row in rows:\n        observations.append({\'ML Model Name\': row[\'ML Model Name\'], \'Observation about model performance\':\n            f"Held-out F1={row[\'F1\']:.4f}, recall={row[\'Recall\']:.4f}, AUC={row[\'AUC\']:.4f}. " + explanations[row[\'ML Model Name\']]})\n    observations.append({\'ML Model Name\': \'Overall winner\', \'Observation about model performance\':\n                         f\'{winner}: selected by highest mean training-only 5-fold CV F1, with CV AUC as tie-breaker. Test scores are a separate final evaluation.\'})\n    readme = f\'\'\'# Cell Nucleus Classification - ML Assignment 2\n\n## a. Problem statement\nCompare five classifiers for malignant versus benign breast-mass classification using cell-nucleus measurements. Educational experiment only.\n\n## b. Dataset description\nUCI Breast Cancer Wisconsin (Diagnostic): 569 instances, 30 numerical input features, no missing values. Target: 1=malignant (212), 0=benign (357). ID is excluded.\nSource: {SOURCE}\nCitation: Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). Breast Cancer Wisconsin (Diagnostic). UCI. https://doi.org/10.24432/C5DW2B. License: CC BY 4.0.\nThe default loader uses sklearn\'s packaged copy; --data accepts original UCI wdbc.data.\nSplit: stratified 80/20, seed {SEED}; {len(X_train)} training and {len(X_test)} test rows. Scaling occurs inside pipelines fitted only on training data. Five-fold CV uses training data only. Model settings are predefined, not optimized on the test set.\n\n## c. Github Repository Link\nTODO: Add your actual GitHub repository URL.\nLive Streamlit app: TODO: Add your deployed app URL.\n\n## d. Models used\nAccuracy, AUC, precision, recall, F1 and MCC below use the identical held-out test set. Malignant is positive. AUC uses probability scores, not predicted labels. Precision/recall/F1 are binary, with zero_division=0.\n\n{markdown_table(metrics)}\n\n### Observations about model performance\n{markdown_table(pd.DataFrame(observations))}\n\n### Training-only cross-validation\n{markdown_table(cv_results)}\n\n## Run locally and on BITS Lab\n1. Install dependencies: `python -m pip install -r requirements.txt` (on first run, install numpy pandas scikit-learn matplotlib joblib streamlit).\n2. Train: `python app.py --train`.\n3. Start app: `python -m streamlit run app.py`.\n4. Upload test_data.csv: all 30 features plus target, with 0/1 labels.\n5. The app loads saved pipelines and evaluates all five models without fitting uploaded data.\nOnly load trusted locally generated joblib artifacts. Use the same Python minor version and dependency versions in lab/cloud as locally; retrain if versions change.\n\n## Repository contents\napp.py (training + app source), requirements.txt, README.md, test_data.csv, model/*.joblib, model/metadata.json, data/dataset.csv, results/metrics.csv, results/cross_validation.csv, confusion plots and classification reports.\n\n## Streamlit Community Cloud\nPush the required files to your GitHub repository. Create an app in Streamlit Community Cloud, choose the repository and branch, and set app.py as the main file. Configure a compatible Python version. Verify CSV upload, model selection, metrics and confusion matrix on the live URL.\n\n## Final submission - action required\nExecute the project on BITS Virtual Lab and capture your own screenshot. Review the code and replace these generated observations with your own interpretation. Add real GitHub and live app links. Create one PDF in this order: GitHub link, live app link, BITS execution screenshot, full finalized README content. Check both clickable links and submit by 8 November 2026, 23:59.\nAI is a learning aid; understand and adapt the implementation and follow your institution\'s originality requirements. No lab screenshot, deployment or submission is performed by this script.\n\'\'\'\n    (ROOT / \'README.md\').write_text(readme, encoding=\'utf-8\')\n    print(metrics.to_string(index=False, float_format=lambda x: f\'{x:.4f}\'))\n    print(f\'\\nTraining-only CV selection: {winner}\')\n    print(f\'Files saved beside script: {ROOT}\')\n    print(\'Next: review README and run python -m streamlit run app.py\')\n\n\ndef app():\n    import streamlit as st\n    import matplotlib.pyplot as plt\n    from sklearn.metrics import ConfusionMatrixDisplay\n    st.set_page_config(page_title=\'Cell Nucleus Model Explorer\', page_icon=\'🔬\', layout=\'wide\')\n    st.title(\'Cell Nucleus Model Explorer\')\n    st.caption(\'Five classifiers · one held-out dataset · malignant = positive class\')\n    metadata_path = ROOT / \'model\' / \'metadata.json\'\n    if not metadata_path.exists():\n        st.info(\'Run python app.py --train first, then restart this app.\')\n        st.stop()\n    metadata = json.loads(metadata_path.read_text())\n    if metadata[\'sklearn_version\'] != sklearn.__version__:\n        st.error(\'scikit-learn version differs from training. Install the generated requirements or retrain with this environment.\')\n        st.stop()\n    st.sidebar.header(\'Evaluation controls\')\n    selected = st.sidebar.selectbox(\'Classifier\', list(SLUGS))\n    upload = st.sidebar.file_uploader(\'Upload labeled test CSV\', type=[\'csv\'])\n    st.sidebar.caption(\'Required: 30 measurement columns and target (0=benign, 1=malignant). No retraining occurs.\')\n    st.sidebar.download_button(\'Download example test CSV\', (ROOT / \'test_data.csv\').read_bytes(), \'test_data.csv\', \'text/csv\')\n    st.info(\'Educational model comparison. Results describe this dataset and split.\')\n    try:\n        frame = pd.read_csv(upload) if upload is not None else pd.read_csv(ROOT / \'test_data.csv\')\n        required = metadata[\'features\'] + [TARGET]\n        if len(frame) == 0 or len(frame) > 10000:\n            raise ValueError(\'Provide between 1 and 10,000 test records.\')\n        if set(frame.columns) != set(required) or len(frame.columns) != len(required):\n            raise ValueError(\'CSV must contain exactly the feature columns in the example CSV plus target; no ID or index column.\')\n        frame = frame[required].apply(pd.to_numeric, errors=\'raise\')\n        if not np.isfinite(frame.to_numpy()).all():\n            raise ValueError(\'Missing or infinite values are not accepted.\')\n        if not frame[TARGET].isin([0, 1]).all():\n            raise ValueError(\'target must contain only 0 (benign) or 1 (malignant).\')\n        X, y = frame[metadata[\'features\']], frame[TARGET].astype(int)\n        results, details = [], {}\n        for name, slug in SLUGS.items():\n            saved = joblib.load(ROOT / \'model\' / f\'{slug}.joblib\')\n            values, predicted, probabilities = evaluate(saved, X, y)\n            results.append({\'ML Model Name\': name, **values})\n            details[name] = (predicted, probabilities)\n    except Exception as error:\n        st.error(f\'Cannot evaluate: {error}\')\n        st.stop()\n    st.write(f\'Evaluating **{len(frame)} records** from \' + (\'your uploaded CSV.\' if upload is not None else \'the saved held-out test set.\'))\n    if y.nunique() < 2:\n        st.warning(\'AUC is undefined because this CSV contains only one target class.\')\n    comparison = pd.DataFrame(results).set_index(\'ML Model Name\')\n    st.subheader(selected)\n    for column, (metric, value) in zip(st.columns(6), comparison.loc[selected].items()):\n        column.metric(metric, \'N/A\' if pd.isna(value) else f\'{value:.4f}\')\n    left, right = st.columns(2)\n    predictions, probabilities = details[selected]\n    with left:\n        fig, ax = plt.subplots(figsize=(5, 4))\n        ConfusionMatrixDisplay(confusion_matrix(y, predictions, labels=[0, 1]), display_labels=[\'Benign\', \'Malignant\']).plot(ax=ax, cmap=\'Blues\', colorbar=False)\n        ax.set_title(\'Actual versus predicted diagnosis\')\n        fig.tight_layout()\n        st.pyplot(fig)\n        plt.close(fig)\n    with right:\n        st.subheader(\'Classification report\')\n        report = classification_report(y, predictions, labels=[0, 1], target_names=[\'Benign\', \'Malignant\'], output_dict=True, zero_division=0)\n        st.dataframe(pd.DataFrame(report).T, use_container_width=True)\n    st.subheader(\'All five models on the same test data\')\n    st.dataframe(comparison.style.format(\'{:.4f}\', na_rep=\'N/A\'), use_container_width=True)\n    st.download_button(\'Download comparison metrics\', comparison.to_csv().encode(), \'comparison_metrics.csv\', \'text/csv\')\n    exported = frame.copy()\n    exported[\'predicted_target\'] = predictions\n    exported[\'malignant_probability\'] = probabilities\n    st.download_button(\'Download selected model predictions\', exported.to_csv(index=False).encode(), \'predictions.csv\', \'text/csv\')\n    with st.expander(\'Dataset, method and model selection\'):\n        st.write(f"Source: {SOURCE}")\n        st.write(f"Training rows: {metadata[\'train_rows\']}; held-out rows: {metadata[\'test_rows\']}; split seed: {SEED}.")\n        st.write(f"Training-only 5-fold CV F1 selection: {metadata[\'selected_model\']}. Uploaded data is evaluated only; it is never used to fit models.")\n        st.dataframe(frame.head())\n\n\nif __name__ == \'__main__\':\n    if \'--train\' in sys.argv:\n        parser = argparse.ArgumentParser(description=__doc__)\n        parser.add_argument(\'--train\', action=\'store_true\')\n        parser.add_argument(\'--data\', type=Path, help=\'Optional original UCI wdbc.data\')\n        args = parser.parse_args()\n        train(args.data)\n    else:\n        app()\n'
(PROJECT / "app.py").write_text(APP_SOURCE, encoding="utf-8")
print("Created", PROJECT / "app.py")


## 3. Train all five models
Choose whether to read original UCI `data/wdbc.data`. If unavailable, the code uses sklearn's packaged copy of the same dataset. Scaling is fitted inside training pipelines. An identical stratified test split evaluates all five models; training-only cross-validation selects a model without tuning on test results.

**This overwrites generated README, requirements, models, test CSV and results.** Set `RUN_TRAINING=False` to inspect existing results only.

In [ ]:
import subprocess
RUN_TRAINING = True
USE_ORIGINAL_UCI_FILE = (PROJECT / "data" / "wdbc.data").exists()
if RUN_TRAINING:
    command = [sys.executable, str(PROJECT / "app.py"), "--train"]
    if USE_ORIGINAL_UCI_FILE:
        command += ["--data", str(PROJECT / "data" / "wdbc.data")]
    result = subprocess.run(command, cwd=PROJECT, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError("Training failed; inspect the error above.")
else:
    print("Using previously generated results.")


## 4. Dataset description and validation
Target: 0=benign, 1=malignant. ID is excluded. Only the held-out test CSV is uploaded to the app.

In [ ]:
data = pd.read_csv(PROJECT / "data" / "dataset.csv")
test = pd.read_csv(PROJECT / "test_data.csv")
print("Full dataset:", data.shape)
print("Input features:", data.shape[1] - 1)
print("Missing values:", data.isna().sum().sum())
print("Target counts:", data["target"].value_counts().to_dict())
print("Held-out test rows:", len(test))
display(data.head())


## 5. All required evaluation metrics
AUC uses malignant probability, not hard predictions. Precision, recall and F1 use malignant as the positive class. Compare metrics together rather than assuming accuracy alone identifies the best model.

In [ ]:
metrics = pd.read_csv(PROJECT / "results" / "metrics.csv")
display(metrics.style.format({c: "{:.4f}" for c in metrics.columns[1:]}))
print("Training-only cross-validation:")
display(pd.read_csv(PROJECT / "results" / "cross_validation.csv"))


## 6. Confusion matrices and classification reports
Confusion matrices use actual classes on rows and predicted classes on columns. Discuss malignant false negatives as well as overall performance.

In [ ]:
import json
names = {"logistic_regression": "Logistic Regression", "decision_tree": "Decision Tree",
         "knn": "KNN", "naive_bayes": "Naive Bayes", "svm": "Support Vector Machine"}
for slug, name in names.items():
    display(Markdown("### " + name))
    display(Image(filename=str(PROJECT / "results" / f"{slug}_confusion.png")))
    report = json.loads((PROJECT / "results" / f"{slug}_report.json").read_text())
    display(pd.DataFrame(report).T)


## 7. Read and finalize the README
The README includes the required problem statement, dataset description, repository link placeholder, model table and observation table. Replace placeholders and revise interpretations in your own words. Update README commands if necessary. Do not rerun training afterward without backing up your edits.

In [ ]:
display(Markdown((PROJECT / "README.md").read_text(encoding="utf-8")))


## 8. Start and test Streamlit
Open Anaconda Prompt in this project folder and run:

```bash
python -m streamlit run app.py
```

Test default data, upload `test_data.csv`, switch among all five models and confirm six metrics, confusion matrix, classification report and all-model comparison. The app evaluates saved pipelines without training uploaded data.

**Keep the test file in CSV format.** An `.xls` extension is not required; do not convert it to an Excel workbook.

## 9. GitHub, BITS Lab and deployment
1. Copy this project to BITS Virtual Lab, install compatible dependencies and run the training command there. Capture one genuine screenshot showing lab context and successful execution.
2. Commit `app.py`, `requirements.txt`, finalized `README.md`, `test_data.csv`, `model/`, and source notebook. Include data/results if desired. Maintain genuine development commits.
3. Deploy the GitHub repository on Streamlit Community Cloud using `app.py`. Choose a Python version compatible with generated dependencies.
4. Verify the live URL and all required interactions. Update README with actual repository and application URLs.

## 10. Create the required submission PDF
The final PDF must contain, in this order:
1. Clickable GitHub repository URL.
2. Clickable live Streamlit application URL.
3. One BITS Virtual Lab execution screenshot.
4. The **entire finalized README**, including both comparison and observation tables.

An easy method is to create a Word document in this order, paste the full rendered README, insert your lab screenshot, and use **Save As → PDF**. Open the PDF and check links, table widths, screenshot legibility and completeness.

For a separate record of this notebook, Jupyter's **Save and Export Notebook As → HTML**, then the browser's **Print → Save as PDF**, can capture executed cells and outputs. That notebook export alone does not replace the specifically ordered submission PDF above.

### Personal review before submission
- Explain each model and the reason for scaling.
- Explain all six metrics and the positive-class mapping.
- Explain training-only CV selection versus held-out test ranking.
- Write your own observations and review the app design.
- Verify both links, screenshot authenticity and full README inclusion.
